# Finding the buildings: choosing the SAM 3 prompts and thresholds

Before a building can be graded, it has to be found and cut out of the photo. We first trained a single detector (RF-DETR) to find buildings and classify them as building, damaged or rubble in one step. It found only 24% of the buildings and drew 8.7–11.6 wrong boxes per photo, and neither a larger model, looser matching nor tiling helped. So the job was split in two: SAM 3 [Carion et al. 2025] finds the buildings, and a separate classifier grades each crop.

SAM 3 segments every instance of a concept described by a short text phrase, without training on our photos. What it finds depends on the phrase and on the score threshold. This notebook describes how they were chosen, with the numbers from the study. It also lets you repeat each step, or try your own prompts.

**Final settings** (`configs/detect.yaml`):

| Prompt | Threshold | Mainly finds |
|---|---|---|
| building reduced to rubble | 0.60 | total collapse (E–F) |
| ruins of a collapsed building | 0.55 | collapsed remains (D–F) |
| damaged standing building | 0.50 | standing damage (B–D) |
| building | 0.50 | intact and other buildings (A–C) |

The four prompts are run with a base confidence of 0.20 and kept above their own threshold. Boxes from different prompts that overlap with IoU > 0.40 are treated as one building, keeping the higher score. Boxes smaller than 4% of the photo width, 5% of its height or 0.4% of its area are dropped.

## Ground truth and scoring

- **Photos:** 361 photos with hand-corrected building boxes, split 255 / 54 / 52 (train / validation / test). Every box carries an A–F grade: 742 / 153 / 155 boxes. Machinery and people were not counted.
- **Matching:** SAM 3 only says "building here", so any predicted box may match any reference box. Matching is one-to-one (Hungarian), and a box counts as found at IoU ≥ 0.5.
- **Main score:** F2, which weights recall four times as much as precision. A missed building costs more than an extra box, because every box is reviewed by hand anyway. Recall was also computed per grade, to see which phrase finds which kind of damage.
- **Discipline:** every choice was made on the 54 validation photos. The 52 test photos were used once, at the end.
- **Caching:** each phrase was run once per photo at a confidence of 0.20, and the answers were saved. All thresholds and combinations below were compared offline from these saved answers.

## Step 1: one phrase at a time

Twenty phrases were tested, in six groups:

| Group | Phrases |
|---|---|
| building | building · damaged building · heavily damaged building · destroyed building |
| collapse | partially collapsed building · collapsed building |
| ruins | ruined building · building ruins · ruins of a collapsed building · collapsed remains of a building |
| structure | destroyed structure · collapsed structure · demolished building |
| rubble | destroyed building rubble pile · building reduced to rubble · pile of rubble from a collapsed building · collapsed building footprint |
| pile | rubble pile · debris pile · building debris pile |

Best single phrases (validation, each at its best threshold):

| Phrase | Precision | Recall | F2 |
|---|---|---|---|
| ruins of a collapsed building | 0.62 | 0.48 | 0.50 |
| destroyed structure | 0.70 | 0.45 | 0.49 |
| destroyed building | 0.65 | 0.42 | 0.45 |
| building | 0.31 | 0.44 | 0.41 |
| heavily damaged building | 0.68 | 0.36 | 0.40 |

No single phrase found every kind of building. Collapse phrases found D–F but missed intact and lightly damaged buildings. `building` did the opposite: it found A–C, but none of E and F. The phrases complement each other.

## Step 2: combining phrases

From the saved answers, 1,350 combinations were scored: 20 single phrases, 190 pairs and 1,140 triples, at a common threshold with duplicates removed. The best combination by F2 was *building reduced to rubble + damaged building + destroyed building rubble pile*: precision 0.53, recall 0.65, F2 0.63 (100 found, 88 wrong, 53 missed). Adding `building` raised recall to 0.76, but only 36% of the boxes were then real buildings, about four wrong boxes per photo.

## Step 3: thresholds per phrase, and the phrases for standing buildings

Collapse phrases also match amorphous debris, so they need a stricter threshold than phrases for standing buildings. About 10,000 settings of per-phrase thresholds and NMS were compared in four sweeps (3,024, 2,808, 2,112 and 2,058 settings). Four phrases for standing buildings were added along the way (*standing building, intact standing building, damaged standing building, partially damaged standing building*). *Damaged standing building* raised the recall of grade C to 0.82.

Two operating points were compared on validation (54 photos, 153 buildings):

| | Precision knee | **More C/D (chosen)** |
|---|---|---|
| Thresholds: rubble / ruins / damaged standing | 0.75 / 0.55 / 0.45 | 0.60 / 0.55 / 0.50 |
| Boxes (correct / wrong) | 66 (59 / 7) | 82 (69 / 13) |
| Precision | 0.894 | 0.841 |
| Recall, all | 0.386 | 0.451 |
| Recall C / D | 0.706 / 0.654 | 0.824 / 0.654 |
| Recall A / B | 0.175 / 0.385 | 0.325 / 0.538 |
| Wrong boxes per photo | 0.13 | 0.24 |

"More C/D" finds 10 more buildings for 6 more wrong boxes. It finds more A, B and C, and loses nothing on D–F. A fourth, general phrase added only wrong boxes at this stage, not more C or D.

## Step 4: looking at the errors

Strict recall (0.45) suggested that SAM 3 missed half of the buildings. The error galleries showed something else. In a rubble field that the reference splits into two to four buildings, SAM 3 often drew one box around the whole field. Strict one-to-one matching counts that as one wrong box plus several misses, although the damage was found.

Two coverage measures were therefore added. A building counts as *found* if at least half of its box is covered by predicted boxes. *Area precision* is the share of predicted area that lies on a reference building.

| Validation, More C/D | Strict | Coverage |
|---|---|---|
| Recall | 0.45 | 0.71 (found at ≥ 50% coverage; 0.77 / 0.68 at 25% / 75%) |
| Precision | 0.84 | 0.92 (area precision) |

Found per grade (≥ 50% covered): A 0.48 · B 0.77 · C 0.94 · D 0.96 · E 0.86 · F 0.61. Strict scoring is pessimistic, because it punishes coarse boxes. Coverage is optimistic, because it ignores whether one box holds one or three buildings. That difference matters, since each crop gets one grade. Both are reported.

## Step 5: one check on the test photos

On the 52 test photos (155 buildings), run once with the chosen setting: precision 0.82, recall 0.39, F2 0.44 (61 found, 13 wrong, 94 missed). Recall for C was 0.74 and for D 0.41 (0.65 on validation); for A it was 0.23. Precision held. The low D recall on test is the main open point of this stage.

## Step 6: settings for building the dataset

The boxes were used to build the classification dataset, and every box was reviewed by hand. In that setting, a missed building costs more than an extra box: deleting a wrong box takes seconds, while a missed building has to be drawn by hand or is lost. So `building` was added back at 0.50. On validation, with `building` (measured at threshold 0.40), the share of buildings found at ≥ 50% coverage rose from 0.71 to 0.86, intact (A) from 0.48 to 0.95, and the covered building area from 0.43 to 0.85. The cost was about 1.6 extra boxes per photo, of which only 15–35% matched a reference building.

**Duplicate removal.** All four prompts can find the same building. NMS at IoU 0.40 merges these duplicates fairly aggressively while keeping neighbouring buildings apart.

**Size filter.** A box is kept only if it is at least 4% of the photo width, 5% of its height and 0.4% of its area. Relative sizes work at any resolution: on a 4000 × 3000 photo this is 160 × 150 px and 48,000 px². The area limit was set on a calibration photo with a large building, a useful partial building (about 3.1% of the area) and a tiny background building (about 0.34%). At 0.3% the tiny building survived; at 0.4% it was removed and the partial building kept. The setting was then checked on rescue scenes, mixed street scenes, dense panoramas, low- and high-resolution city views, and partly cut-off buildings. A stricter rule for boxes touching the image edge removed a valid building, so edge boxes follow the same rule as all others.

**Aerial photos.** A small vision-language model (Qwen 3.5 VL 0.8B, run on Roboflow) labels each photo AERIAL or GROUND_LEVEL with the prompt in `configs/detect.yaml`. Aerial photos get no boxes. It matched the expected answer on 4 of 4 check images (clear aerial, borderline rooftop panorama, ground-level rescue scene, ground-level airfield). The filter is off by default here.

**Review rules.** A box was approved if it showed one distinct building with enough of it visible to grade. It was corrected if a building was missed, the box was too broad or narrow, buildings were merged, or one building was split. It was rejected for aerial views, scenes without a usable building (people, vehicles, tents, interiors), rubble fragments without a building outline, buildings too small or blurred to grade, and crops too incomplete to describe the building.

## Repeating the comparison

The cells below repeat the study's steps on the validation photos. You can use the study's prompts, or add your own in `EXTRA_PROMPTS`. SAM 3 runs once per prompt and photo; everything after that is computed from the saved answers.

Requirements: a GPU, a Hugging Face account with access to `facebook/sam3`, and a free Roboflow API key to download the reference boxes. In the study SAM 3 ran on Roboflow, and scores from the Hugging Face version may differ slightly, so check the thresholds visually before relying on them.

In [ ]:
# On Colab: install the package (with SAM 3 support) and get the configuration files.
!pip -q install "insarag-triage[detect,roboflow] @ git+https://github.com/leeelyehezkel/insarag-damage-triage"
!git clone -q https://github.com/leeelyehezkel/insarag-damage-triage repo 2>/dev/null || true

In [ ]:
from pathlib import Path
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from insarag_triage import detect as D, data as DATA

REPO = Path('repo') if Path('repo').exists() else Path('.')
cfg = D.load_config(REPO / 'configs' / 'detect.yaml')
STUDY_PROMPTS = dict(cfg['prompts'])          # prompt -> threshold
STUDY_TRIED = ['ruins of a collapsed building', 'destroyed structure', 'destroyed building', 'heavily damaged building']
EXTRA_PROMPTS = []                            # your own phrases, e.g. ['collapsed house', 'damaged apartment block']
PROMPTS = list(dict.fromkeys(list(STUDY_PROMPTS) + STUDY_TRIED + EXTRA_PROMPTS))

gt_root = DATA.download(DATA.load_config(REPO / 'configs' / 'data.yaml'), 'detection_gt', 'work/roboflow', fmt='coco')
gt_val, gt_test = DATA.load_coco_boxes(gt_root / 'valid'), DATA.load_coco_boxes(gt_root / 'test')
print(f"validation: {gt_val.photo.nunique()} photos, {len(gt_val)} buildings · test: {gt_test.photo.nunique()} photos, {len(gt_test)} buildings")
gt_val.grade.value_counts().sort_index()

In [ ]:
from huggingface_hub import login
login()                                       # SAM 3 is gated: accept its terms on Hugging Face first
sam, sam_proc = D.load_sam3(cfg['model_id'])

### Run every prompt once on the validation photos

About one second per prompt and photo on a T4. The answers are saved, so this only needs to run once.

In [ ]:
CACHE = Path('work/sam3_valid_detections.csv')
val_photos = sorted(gt_val.path.unique())
if CACHE.exists():
    dets = pd.read_csv(CACHE)
    new = [p for p in PROMPTS if p not in set(dets.prompt)]
    if new:
        dets = pd.concat([dets, D.detect_many(sam, sam_proc, val_photos, new, cfg['base_confidence'])], ignore_index=True)
else:
    dets = D.detect_many(sam, sam_proc, val_photos, PROMPTS, cfg['base_confidence'])
dets.to_csv(CACHE, index=False)
D.threshold_table(dets, [0.3, 0.4, 0.5, 0.6, 0.7])   # boxes per photo, per prompt and threshold

### Single prompts and combinations

In [ ]:
rows = []
for p in PROMPTS:
    for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
        s = D.score(D.select(dets, {p: t}, cfg['nms_iou']), gt_val)
        rows.append(dict(prompt=p, threshold=t, **{k: s.get(k) for k in ['precision', 'recall', 'f2', 'recall_C', 'recall_D']}))
single = pd.DataFrame(rows)
single.loc[single.groupby('prompt').f2.idxmax()].sort_values('f2', ascending=False).round(3)

In [ ]:
combos = D.search_combinations(dets, gt_val, PROMPTS, threshold=0.5, max_size=3, nms_iou=cfg['nms_iou'])
combos.head(10).round(3)

### The chosen setting, strict and by coverage

In [ ]:
CHOSEN = dict(STUDY_PROMPTS)                  # change thresholds or prompts here, e.g. CHOSEN['building'] = 0.6
sel = D.select(dets, CHOSEN, cfg['nms_iou'], cfg['size_filter'])
strict = D.score(sel, gt_val)
cover = D.region_scores(sel, gt_val)
print('strict  :', {k: round(v, 3) for k, v in strict.items() if isinstance(v, float)})
print('coverage:', {k: round(v, 3) for k, v in cover.items()})

### Visual check

Counts and scores don't show *which* buildings are missed, so look through the photos. Coloured boxes are SAM 3's, one colour per prompt with the score. Dashed white boxes are the reference buildings, and missed ones are outlined in red. Look in particular for:

- collapsed buildings and debris heaps that no prompt finds;
- boxes around non-buildings (vehicles, machinery, rubble fragments);
- one box over several buildings, or one building split into several boxes;
- buildings that are too small or blurred to grade, which the size filter should remove.

In [ ]:
m = D.match(sel, gt_val)
worst = m.groupby('photo').found.mean().sort_values().index[:6]      # photos with the most misses first
fig, axs = plt.subplots(2, 3, figsize=(18, 10))
for ax, photo in zip(axs.flat, worst):
    path = gt_val.loc[gt_val.photo == photo, 'path'].iloc[0]
    D.show_detections(path, sel, gt_val, ax=ax)
    for _, r in m[(m.photo == photo) & ~m.found].iterrows():
        ax.add_patch(plt.Rectangle((r.x, r.y), r.w, r.h, fill=False, lw=2.5, color='red'))
plt.tight_layout(); plt.show()

In [ ]:
# One photo at any threshold
PHOTO, PROMPT, THRESHOLD = val_photos[0], 'building', 0.5
one = D.select(dets, {PROMPT: THRESHOLD}, cfg['nms_iou'])
plt.figure(figsize=(10, 7)); D.show_detections(PHOTO, one, gt_val); plt.show()

### Test once, and save the settings

Only after the settings are fixed: run them once on the test photos. The saved file is used by `01_photos_to_grades` instead of the study's settings.

In [ ]:
RUN_TEST = False
if RUN_TEST:
    test_dets = D.detect_many(sam, sam_proc, sorted(gt_test.path.unique()), list(CHOSEN), cfg['base_confidence'])
    test_sel = D.select(test_dets, CHOSEN, cfg['nms_iou'], cfg['size_filter'])
    print('test strict  :', {k: round(v, 3) for k, v in D.score(test_sel, gt_test).items() if isinstance(v, float)})
    print('test coverage:', {k: round(v, 3) for k, v in D.region_scores(test_sel, gt_test).items()})
cfg['prompts'] = CHOSEN
Path('detect_custom.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))